
# Step 1: Qwen2.5-7B-Instruct (base) — Handwriting Stroke Inference

- Model: `Qwen/Qwen2.5-7B-Instruct` loaded in 8-bit on a Colab T4 (~9 GB VRAM).
- Task: emit JSON stroke actions on a 100x100 canvas, max 15 strokes, goal 90% pixel coverage of a target capital letter.
- Eval: 2 characters (`L` easy, `B` hard). Numbers only — no canvas images.
- Output of this notebook is the baseline you will compare against the GRPO LoRA (`abhijeetmishra101/Qwen2.5-7B-Handwriting-GRPO`) in Step 2.

In [1]:
!pip install -q -U --force-reinstall "pillow>=11.0.0,<12.0.0" "pydantic>=2.0.0,<=2.12.3"
!pip install -q -U transformers accelerate bitsandbytes opencv-python-headless

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 462.4/462.4 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00


In [2]:
#import json, re, textwrap, time
from typing import Any, Optional

import cv2
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available(), "Switch Colab runtime to GPU (T4)."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")


GPU: Tesla T4
VRAM: 14.6 GB


In [3]:
FONT_SIZE = 80
BRUSH_WIDTH = 8
INTEGRITY_THRESHOLD = 0.60
MAX_DRAWN_MULTIPLIER = 1.7

_FONT_CANDIDATES = [
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
    "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
    "/usr/share/fonts/truetype/freefont/FreeSansBold.ttf",
]

def _load_font(size: int) -> ImageFont.FreeTypeFont:
    for path in _FONT_CANDIDATES:
        try:
            return ImageFont.truetype(path, size)
        except OSError:
            continue
    raise RuntimeError("No bold TrueType font found on this Colab image.")

_FONT = _load_font(FONT_SIZE)

def render_target_character(character: str) -> np.ndarray:
    """Render a single capital letter into a 100x100 binary array (1=ink)."""
    img = Image.new("L", (100, 100), color=0)
    draw = ImageDraw.Draw(img)
    bbox = draw.textbbox((0, 0), character, font=_FONT)
    x_off = (100 - (bbox[2] - bbox[0])) // 2 - bbox[0]
    y_off = (100 - (bbox[3] - bbox[1])) // 2 - bbox[1]
    draw.text((x_off, y_off), character, font=_FONT, fill=255)
    arr = np.array(img, dtype=np.uint8)
    return (arr >= 128).astype(np.uint8)

def compute_character_bbox(character: str) -> tuple[int, int, int, int]:
    arr = render_target_character(character)
    rows = np.any(arr, axis=1)
    cols = np.any(arr, axis=0)
    y_min, y_max = int(np.where(rows)[0][0]), int(np.where(rows)[0][-1])
    x_min, x_max = int(np.where(cols)[0][0]), int(np.where(cols)[0][-1])
    return x_min, y_min, x_max, y_max

def _flood_fill_interior(char_matrix: np.ndarray) -> np.ndarray:
    inverted = (1 - char_matrix).astype(np.uint8)
    h, w = inverted.shape
    flood = inverted.copy()
    fill_mask = np.zeros((h + 2, w + 2), dtype=np.uint8)
    cv2.floodFill(flood, fill_mask, (0, 0), 2)
    interior = (flood == 1).astype(np.uint8)
    return (char_matrix | interior).astype(np.uint8)

DISQUALIFICATION_MASKS: dict[str, list[np.ndarray]] = {}

def _build_disqualification_masks() -> None:
    diff_pairs = {"C": "O", "S": "8", "G": "O"}
    for good, bad in diff_pairs.items():
        g = render_target_character(good)
        b = render_target_character(bad)
        mask = np.clip(b.astype(np.int16) - g.astype(np.int16), 0, 1).astype(np.uint8)
        if mask.sum() > 0:
            DISQUALIFICATION_MASKS[good] = [mask]
    for ch in ["A", "B", "D", "O", "P", "Q", "R"]:
        target = render_target_character(ch)
        filled = _flood_fill_interior(target)
        interior = np.clip(filled.astype(np.int16) - target.astype(np.int16), 0, 1).astype(np.uint8)
        if ch == "B":
            n_labels, labels = cv2.connectedComponents(interior)
            masks = []
            for label_id in range(1, n_labels):
                comp = (labels == label_id).astype(np.uint8)
                if comp.sum() > 50:
                    masks.append(comp)
            if masks:
                DISQUALIFICATION_MASKS[ch] = masks
        elif interior.sum() > 0:
            DISQUALIFICATION_MASKS[ch] = [interior]

_build_disqualification_masks()

def draw_action(action: dict) -> np.ndarray:
    """Return a 100x100 binary matrix with the stroke painted."""
    canvas = np.zeros((100, 100), dtype=np.int32)
    w = BRUSH_WIDTH
    t = action.get("action_type", "line")
    x1, y1 = int(action.get("x1", 0)), int(action.get("y1", 0))
    if t == "circle" and action.get("radius") is not None:
        cv2.circle(canvas, (x1, y1), int(action["radius"]), 1, w)
    elif t == "ellipse" and action.get("rx") is not None and action.get("ry") is not None:
        cv2.ellipse(canvas, (x1, y1), (int(action["rx"]), int(action["ry"])),
                    0, 0, 360, 1, w)
    elif (t == "curve" and action.get("x2") is not None and action.get("y2") is not None
          and action.get("x3") is not None and action.get("y3") is not None):
        x2, y2 = int(action["x2"]), int(action["y2"])
        x3, y3 = int(action["x3"]), int(action["y3"])
        cx = 2 * x3 - 0.5 * x1 - 0.5 * x2
        cy = 2 * y3 - 0.5 * y1 - 0.5 * y2
        ts = np.linspace(0, 1, 50)
        x = (1 - ts) ** 2 * x1 + 2 * (1 - ts) * ts * cx + ts ** 2 * x2
        y = (1 - ts) ** 2 * y1 + 2 * (1 - ts) * ts * cy + ts ** 2 * y2
        pts = np.stack((x, y), axis=1).astype(np.int32).reshape((-1, 1, 2))
        cv2.polylines(canvas, [pts], isClosed=False, color=1, thickness=w)
    elif t == "line" and action.get("x2") is not None and action.get("y2") is not None:
        cv2.line(canvas, (x1, y1), (int(action["x2"]), int(action["y2"])), 1, w)
    return canvas

print("Built integrity masks for:", sorted(DISQUALIFICATION_MASKS.keys()))

Built integrity masks for: ['A', 'B', 'C', 'D', 'G', 'O', 'P', 'Q', 'R', 'S']


## Watch episode


In [5]:
MAX_STEPS = 15
import textwrap

SYSTEM_PROMPT = textwrap.dedent(f"""
    You are drawing capital letters on a 100x100 pixel canvas.

    Canvas coordinate system:
    - x: 0 (left) -> 99 (right)
    - y: 0 (top)  -> 99 (bottom)
    - Origin (0,0) is the TOP-LEFT corner

    You can draw lines, curves, circles, and ellipses using these action types:
    - "line": Straight line from (x1,y1) to (x2,y2).
    - "curve": A curve starting at (x1,y1), ending at (x2,y2), and passing through a midpoint at (x3,y3).
    - "circle": A circle centered at (x1,y1) with given `radius`.
    - "ellipse": An upright oval centered at (x1,y1) with horizontal radius `rx` and vertical radius `ry`.

    Rules:
    - You have at most {MAX_STEPS} actions per episode.
    - Goal: cover 90% of the target character's white pixels.
    - INK PENALTY: You will fail the episode instantly if you draw more than {MAX_DRAWN_MULTIPLIER}x the target's total pixels. Do not waste ink!
    - Coordinates are integers 0-99.

    SHAPE INTEGRITY - protected regions you must NOT fill in:
    - A: inner triangle hole (the counter between the two legs and crossbar)
    - B: two enclosed lobe holes (upper and lower bumps)
    - D: interior of the D bowl (semicircle counter - outline only, like O)
    - O: circle interior (do not fill the hole)
    - P: bowl interior (do not fill the hole in the loop)
    - R: bowl interior (do not fill the hole above the diagonal leg)
    - C: right-side opening (do not close it - that would make O)
    - S: two bridge gaps (do not connect the loops - that would make 8)
    - G: right-side opening (do not close it - that would make O)
    - Q: circle interior (same as O - do not fill the hole; draw the tail as a separate stroke)
    If you cover more than 60% of a protected region the episode ends immediately
    with reward=0 and integrity_violated=True. Plan your strokes to follow the
    character outline only, never filling in holes or closing open gaps.

    You MUST respond with a valid JSON object and nothing else. No markdown, no explanation outside the JSON.

    Sample Invocations:
    - To draw a vertical line on the left side:
      {{"reasoning": "Drawing the vertical spine of the letter D", "action_type": "line", "x1": 20, "y1": 10, "x2": 20, "y2": 90}}

    - To draw a curved right side of a D (starts top, ends bottom, bows out to x=80):
      {{"reasoning": "Drawing the curved belly of the letter D", "action_type": "curve", "x1": 20, "y1": 10, "x2": 20, "y2": 90, "x3": 80, "y3": 50}}

    - To draw a perfect circle for an O:
      {{"reasoning": "Drawing the letter O using a circle centered in the canvas", "action_type": "circle", "x1": 50, "y1": 50, "radius": 40}}

    - To draw a tall, narrow oval:
      {{"reasoning": "Drawing a tall vertical ellipse", "action_type": "ellipse", "x1": 50, "y1": 50, "rx": 20, "ry": 40}}
""").strip()

def build_user_prompt(target_character, step, strokes_remaining, match_percentage,
                     last_pixels_matched, last_pixels_wasted, ink_remaining,
                     last_reward, history, last_integrity_violated=False, char_bbox=None):
    history_block = "\n".join(history[-5:]) if history else "None yet"
    integrity_warning = ("\n[!] LAST ACTION VIOLATED SHAPE INTEGRITY - episode ended with reward=0."
                        if last_integrity_violated else "")
    bbox_info = (f"Bounding Box: x=[{char_bbox[0]}->{char_bbox[2]}], y=[{char_bbox[1]}->{char_bbox[3]}]\n"
                 if char_bbox else "")
    return textwrap.dedent(f"""
        Draw the capital letter: {target_character}
        {bbox_info}
        Step: {step} / {MAX_STEPS}
        Actions remaining: {strokes_remaining}
        Current coverage: {match_percentage:.1%} (goal: 90%)

        FEEDBACK ON LAST ACTION:
        - Pixels matched: {last_pixels_matched}
        - Pixels wasted (missed target): {last_pixels_wasted}
        - Ink remaining before failure: {ink_remaining}{integrity_warning}

        Action history:
        {history_block}

        Plan your next action to maximise coverage of '{target_character}' without wasting ink or violating shape integrity.
    """).strip()

print("System prompt length:", len(SYSTEM_PROMPT), "chars")

System prompt length: 2612 chars


## Plots


In [6]:
BASE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
import time
bnb = BitsAndBytesConfig(load_in_8bit=True)

print(f"Loading {BASE_MODEL_ID} in 8-bit ... (~6 minutes first time)")
t0 = time.time()
tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print(f"Loaded in {time.time() - t0:.1f}s")
print("VRAM allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

Loading Qwen/Qwen2.5-7B-Instruct in 8-bit ... (~6 minutes first time)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loaded in 469.8s
VRAM allocated: 8.11 GB


In [11]:
TEMPERATURE = 0.7
MAX_NEW_TOKENS = 256
import re
import json
_JSON_FENCE_RE = re.compile(r"```(?:json)?\s*(\{.*?\})\s*```", re.DOTALL)
_JSON_OBJECT_RE = re.compile(r"\{.*\}", re.DOTALL)

def _extract_json(raw: str) -> dict:
    if not raw:
        return {}
    m = _JSON_FENCE_RE.search(raw)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    m = _JSON_OBJECT_RE.search(raw)
    if m:
        try:
            return json.loads(m.group(0))
        except json.JSONDecodeError:
            return {}
    return {}

def _clamp(v, lo, hi):
    try:
        return max(lo, min(hi, int(round(float(v)))))
    except (TypeError, ValueError):
        return None

def _normalize_stroke(d: dict) -> dict:
    out = {
        "reasoning":   d.get("reasoning", ""),
        "action_type": d.get("action_type", "line"),
        "x1": _clamp(d.get("x1", 10), 0, 99) or 10,
        "y1": _clamp(d.get("y1", 10), 0, 99) or 10,
    }
    for k in ("x2", "y2", "x3", "y3"):
        if d.get(k) is not None:
            out[k] = _clamp(d[k], 0, 99)
    for k in ("radius", "rx", "ry"):
        if d.get(k) is not None:
            out[k] = _clamp(d[k], 1, 100)
    if out["action_type"] not in ("line", "curve", "circle", "ellipse"):
        out["action_type"] = "line"
    return out

def get_stroke(messages: list[dict]) -> dict:
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=0.9,
            pad_token_id=tok.eos_token_id,
        )
    raw = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    parsed = _extract_json(raw)
    if not parsed:
        return {"reasoning": "fallback (parse error)", "action_type": "line",
                "x1": 10, "y1": 10, "x2": 90, "y2": 90}
    return _normalize_stroke(parsed)

print("get_stroke() ready")

get_stroke() ready


In [9]:
def _check_integrity(canvas: np.ndarray, character: str) -> bool:
    for mask in DISQUALIFICATION_MASKS.get(character, []):
        zone = int(mask.sum())
        if zone == 0:
            continue
        covered = int((canvas * mask).sum())
        if covered / zone > INTEGRITY_THRESHOLD:
            return True
    return False

def run_episode(character: str, verbose: bool = True) -> dict:
    target = render_target_character(character).astype(np.int32)
    total_target = int(target.sum())
    bbox = compute_character_bbox(character)
    canvas = np.zeros((100, 100), dtype=np.int32)
    max_allowed = int(MAX_DRAWN_MULTIPLIER * total_target)

    history: list[str] = []
    rewards: list[float] = []
    last_matched, last_wasted = 0, 0
    ink_remaining = max_allowed
    last_reward = 0.0
    last_integrity = False
    match_percentage = 0.0
    integrity_violated = False
    steps_taken = 0

    if verbose:
        print(f"\n=== {character} (target_pixels={total_target}, max_ink={max_allowed}) ===")

    for step in range(1, MAX_STEPS + 1):
        strokes_remaining = MAX_STEPS - step + 1
        user_msg = build_user_prompt(
            character, step, strokes_remaining, match_percentage,
            last_matched, last_wasted, ink_remaining, last_reward, history,
            last_integrity_violated=last_integrity, char_bbox=bbox,
        )
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ]
        stroke = get_stroke(messages)

        temp = draw_action(stroke)
        prev_total_matched = int((canvas * target).sum())
        canvas = np.maximum(canvas, temp)
        integrity_violated = _check_integrity(canvas, character)
        total_matched = int((canvas * target).sum())
        last_matched = total_matched - prev_total_matched
        reward = (last_matched / total_target) if total_target > 0 else 0.0
        if integrity_violated:
            reward = 0.0
        match_percentage = total_matched / total_target if total_target > 0 else 0.0

        drawn_this = int(temp.sum())
        on_target_this = int((temp * target).sum())
        last_wasted = max(0, drawn_this - on_target_this)
        total_drawn = int(canvas.sum())
        ink_remaining = max(0, max_allowed - total_drawn)

        rewards.append(reward)
        last_reward = reward
        last_integrity = integrity_violated
        steps_taken = step

        if stroke["action_type"] == "circle":
            astr = f"circle({stroke['x1']},{stroke['y1']},r={stroke.get('radius')})"
        elif stroke["action_type"] == "ellipse":
            astr = f"ellipse({stroke['x1']},{stroke['y1']},rx={stroke.get('rx')},ry={stroke.get('ry')})"
        elif stroke["action_type"] == "curve":
            astr = f"curve({stroke['x1']},{stroke['y1']}->{stroke.get('x2')},{stroke.get('y2')} via {stroke.get('x3')},{stroke.get('y3')})"
        else:
            astr = f"line({stroke['x1']},{stroke['y1']}->{stroke.get('x2')},{stroke.get('y2')})"

        history.append(
            f"Step {step}: {astr} matched={last_matched}px reward={reward:.4f} "
            f"coverage={match_percentage:.1%}{' [INTEGRITY]' if integrity_violated else ''}"
        )
        if verbose:
            tag = "  [INTEGRITY]" if integrity_violated else ""
            print(f"  step {step:>2}: {astr:<50} cov={match_percentage:>6.1%} "
                  f"reward={reward:>6.3f} ink_left={ink_remaining}{tag}")

        done = (
            integrity_violated
            or match_percentage >= 0.90
            or steps_taken >= MAX_STEPS
            or total_drawn > max_allowed
        )
        if done:
            break

    return {
        "character": character,
        "steps": steps_taken,
        "final_coverage": match_percentage,
        "success": match_percentage >= 0.90 and not integrity_violated,
        "integrity_violated": integrity_violated,
        "sum_reward": sum(rewards),
    }

print("run_episode() ready")

run_episode() ready


In [12]:
EVAL_CHARS = ["L", "T", "V", "I", "F"]

rows = []
for ch in EVAL_CHARS:
    rows.append(run_episode(ch, verbose=True))

print("\n" + "=" * 72)
print(f"{'CHAR':<6}{'STEPS':<8}{'COVERAGE':<12}{'SUCCESS':<10}{'INTEGRITY':<12}{'SUM_REWARD':<12}")
print("-" * 72)
for r in rows:
    print(f"{r['character']:<6}{r['steps']:<8}{r['final_coverage']:>7.1%}     "
          f"{str(r['success']):<10}{str(r['integrity_violated']):<12}{r['sum_reward']:<12.4f}")
print("-" * 72)
mean_cov = sum(r["final_coverage"] for r in rows) / len(rows)
succ = sum(1 for r in rows if r["success"])
print(f"BASE  mean_coverage={mean_cov:.1%}  successes={succ}/{len(rows)}  "
      f"integrity_violations={sum(1 for r in rows if r['integrity_violated'])}")
print("=" * 72)

BASE_RESULTS = {"model": "Qwen/Qwen2.5-7B-Instruct", "rows": rows,
                "mean_coverage": mean_cov, "successes": succ}
print("\nSaved to BASE_RESULTS for Step 2 to compare against.")


=== L (target_pixels=921, max_ink=1565) ===


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


  step  1: line(30,22->30,76)                                 cov= 29.9% reward= 0.299 ink_left=1030
  step  2: line(30,22->70,22)                                 cov= 33.7% reward= 0.038 ink_left=678
  step  3: line(70,22->70,76)                                 cov= 38.5% reward= 0.049 ink_left=200
  step  4: line(30,76->70,76)                                 cov= 55.4% reward= 0.168 ink_left=0

=== B (target_pixels=1818, max_ink=3090) ===
  step  1: line(26,22->26,76)                                 cov= 15.1% reward= 0.151 ink_left=2555
  step  2: line(48,22->74,22)                                 cov= 20.2% reward= 0.051 ink_left=2272
  step  3: line(26,44->74,44)                                 cov= 34.2% reward= 0.139 ink_left=1856
  step  4: line(74,22->74,76)                                 cov= 39.4% reward= 0.053 ink_left=1443
  step  5: line(48,66->74,66)                                 cov= 46.2% reward= 0.068 ink_left=1225
  step  6: line(48,55->74,55)                     

In [13]:
import importlib, sys

try:
    importlib.import_module("peft")
    print("peft: already installed")
except Exception:
    !pip install -q -U peft
    importlib.invalidate_caches()
    import peft
    print("peft: installed now", getattr(peft, "__version__", "unknown"))

peft: already installed


In [14]:
from peft import PeftModel

ADAPTER_ID = "abhijeetmishra101/Qwen2.5-7B-Handwriting-GRPO"

print(f"Loading LoRA adapter: {ADAPTER_ID}")
t0 = time.time()

# Wrap the already-loaded base `model` with the LoRA adapter.
# This keeps the same tokenizer + quantization, and swaps behavior via LoRA weights.
model = PeftModel.from_pretrained(model, ADAPTER_ID)
model.eval()

print(f"LoRA attached in {time.time() - t0:.1f}s")
print("Model class:", type(model))

Loading LoRA adapter: abhijeetmishra101/Qwen2.5-7B-Handwriting-GRPO


adapter_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


adapter_model.safetensors:   0%|          | 0.00/162M [00:00<?, ?B/s]

LoRA attached in 8.0s
Model class: <class 'peft.peft_model.PeftModelForCausalLM'>


In [15]:
lora_rows = []
for ch in EVAL_CHARS:
    lora_rows.append(run_episode(ch, verbose=True))

print("\n" + "=" * 88)
print(f"{'CHAR':<6}{'BASE_COV':<12}{'LORA_COV':<12}{'ΔCOV':<10}{'BASE_OK':<10}{'LORA_OK':<10}{'BASE_INT':<10}{'LORA_INT':<10}")
print("-" * 88)

base_by_char = {r["character"]: r for r in BASE_RESULTS["rows"]}
lora_by_char = {r["character"]: r for r in lora_rows}

for ch in EVAL_CHARS:
    b = base_by_char[ch]
    l = lora_by_char[ch]
    dc = l["final_coverage"] - b["final_coverage"]
    print(
        f"{ch:<6}"
        f"{b['final_coverage']:>7.1%}     "
        f"{l['final_coverage']:>7.1%}     "
        f"{dc:>+7.1%}   "
        f"{str(b['success']):<10}{str(l['success']):<10}"
        f"{str(b['integrity_violated']):<10}{str(l['integrity_violated']):<10}"
    )

base_mean = BASE_RESULTS["mean_coverage"]
lora_mean = sum(r["final_coverage"] for r in lora_rows) / len(lora_rows)

base_succ = sum(1 for r in BASE_RESULTS["rows"] if r["success"])
lora_succ = sum(1 for r in lora_rows if r["success"])

base_int = sum(1 for r in BASE_RESULTS["rows"] if r["integrity_violated"])
lora_int = sum(1 for r in lora_rows if r["integrity_violated"])

print("-" * 88)
print(
    f"MEAN  "
    f"{base_mean:>7.1%}     "
    f"{lora_mean:>7.1%}     "
    f"{(lora_mean - base_mean):>+7.1%}   "
    f"succ {base_succ}/{len(EVAL_CHARS)} -> {lora_succ}/{len(EVAL_CHARS)}   "
    f"integrity {base_int} -> {lora_int}"
)
print("=" * 88)

LORA_RESULTS = {"model": ADAPTER_ID, "rows": lora_rows, "mean_coverage": lora_mean, "successes": lora_succ}
print("\nSaved to LORA_RESULTS (and BASE_RESULTS already exists).")


=== L (target_pixels=921, max_ink=1565) ===
  step  1: line(50,22->50,76)                                 cov=  8.8% reward= 0.088 ink_left=1030
  step  2: line(30,22->30,76)                                 cov= 38.7% reward= 0.299 ink_left=495
  step  3: line(30,22->70,22)                                 cov= 42.5% reward= 0.038 ink_left=208
  step  4: line(70,22->70,76)                                 cov= 47.3% reward= 0.049 ink_left=0

=== B (target_pixels=1818, max_ink=3090) ===
  step  1: line(50,22->50,76)                                 cov= 13.4% reward= 0.134 ink_left=2555
  step  2: line(35,22->65,22)                                 cov= 19.5% reward= 0.061 ink_left=2301
  step  3: line(35,50->65,50)                                 cov= 30.6% reward= 0.112 ink_left=2063
  step  4: line(35,50->65,50)                                 cov= 30.6% reward= 0.000 ink_left=2063
  step  5: line(35,50->65,50)                                 cov= 30.6% reward= 0.000 ink_left=2063
  ste